In [1]:
import re
import difflib
import json
from collections import defaultdict
import nltk
from nltk.corpus import words, stopwords, brown
import jellyfish

In [2]:
# --- 1. Dynamic NLP Setup & Configuration ---
nltk.download('words', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('brown', quiet=True)

STOP_WORDS = set(stopwords.words('english'))
FREQ_DIST = nltk.FreqDist(w.lower() for w in brown.words())

def get_phonetic_key(word):
    if not word: return ""
    return jellyfish.metaphone(word.lower())

STANDARD_DICTIONARY = {w.lower() for w in words.words() if w.isalpha()}

REINFORCEMENT_THRESHOLD = 3
INFERENCE_THRESHOLD = 0.005

In [3]:
def extract_window(tokens, target_idx, radius=4):
    start = max(0, target_idx - radius)
    end = min(len(tokens), target_idx + radius + 1)
    window = []
    for i in range(start, end):
        if i == target_idx: continue
        w = tokens[i].group(0).lower()
        if w not in STOP_WORDS:
            window.append(w)
    return window

In [4]:
def learn_from_observation(formatted_text, final_text, memory_db):
    formatted_tokens = list(re.finditer(r'\b\w+\b', formatted_text))
    final_tokens = list(re.finditer(r'\b\w+\b', final_text))
    
    formatted_words = [m.group(0) for m in formatted_tokens]
    final_words = [m.group(0) for m in final_tokens]
    
    sm = difflib.SequenceMatcher(None, [w.lower() for w in formatted_words], [w.lower() for w in final_words])
    changed_indices = set()
    corrections = []
    
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == 'replace' and (i2 - i1) == 1 and (j2 - j1) == 1:
            corrections.append((i1, formatted_words[i1], final_words[j1]))
            changed_indices.add(i1)
            
    for idx, orig_word, new_word in corrections:
        revert_entry_key = next((k for k, v in memory_db.items() if v['canonical_term'] == orig_word), None)
        if revert_entry_key:
            # This is a revert! The user undid our intervention.
            entry = memory_db[revert_entry_key]
            
            # Exponential decay: halve the count instead of flat -2 subtraction
            entry['observation_count'] = entry['observation_count'] // 2
            entry['confidence'] = min(1.0, entry['observation_count'] / REINFORCEMENT_THRESHOLD)
            if entry['confidence'] < 1.0:
                entry['status'] = 'candidate'
            if entry['observation_count'] == 0:
                entry['status'] = 'retired'
                
            # Crucial: Learn from the mistake by adding the context to negative anchors!
            neg_window = extract_window(formatted_tokens, idx)
            for w in neg_window:
                entry['negative_anchors'][w] += 1
            continue
            
        pkey = get_phonetic_key(orig_word)
        canonical = new_word
        db_key = f"{pkey}::{canonical}"
        
        if db_key not in memory_db:
            memory_db[db_key] = {
                'canonical_term': canonical,
                'phonetic_key': pkey,
                'confidence': 0.0,
                'observation_count': 0,
                'ambiguity_risk': orig_word.lower() in STANDARD_DICTIONARY or canonical.lower() in STANDARD_DICTIONARY,
                'positive_anchors': defaultdict(int),
                'negative_anchors': defaultdict(int),
                'status': 'candidate'
            }
            
        entry = memory_db[db_key]
        
        # Uncap the observation count so it can grow indefinitely for analytics
        entry['observation_count'] += 1
        entry['confidence'] = min(1.0, entry['observation_count'] / REINFORCEMENT_THRESHOLD)
        
        if entry['confidence'] >= 1.0:
            entry['status'] = 'active'
            
        if entry['ambiguity_risk']:
            pos_window = extract_window(formatted_tokens, idx)
            for w in pos_window:
                entry['positive_anchors'][w] += 1
                
            for sibling_idx, token in enumerate(formatted_tokens):
                if sibling_idx == idx or sibling_idx in changed_indices:
                    continue
                
                sibling_word = token.group(0).lower()
                if get_phonetic_key(sibling_word) == pkey:
                    neg_window = extract_window(formatted_tokens, sibling_idx)
                    for w in neg_window:
                        entry['negative_anchors'][w] += 1

    return memory_db

In [5]:
def apply_memory(formatted_text, memory_db):
    tokens = list(re.finditer(r'\b\w+\b', formatted_text))
    result = formatted_text
    
    for idx in range(len(tokens)-1, -1, -1):
        token = tokens[idx]
        word = token.group(0)
        pkey = get_phonetic_key(word)
        
        candidates = [entry for key, entry in memory_db.items() if entry['phonetic_key'] == pkey and entry.get('status') == 'active']
        
        if not candidates:
            continue
            
        best_candidate = None
        best_score = -float('inf')
        
        for entry in candidates:
            if not entry['ambiguity_risk']:
                best_candidate = entry
                best_score = float('inf')
                break
            else:
                window = extract_window(tokens, idx)
                score = 0
                for w in window:
                    doc_freq = FREQ_DIST.get(w, 0)
                    weight = 1.0 / (1.0 + doc_freq)
                    pos_count = entry['positive_anchors'].get(w, 0)
                    neg_count = entry['negative_anchors'].get(w, 0)
                    score += weight * (pos_count - neg_count)
                
                if score >= INFERENCE_THRESHOLD and score > best_score:
                    best_candidate = entry
                    best_score = score
                elif score < INFERENCE_THRESHOLD:
                    if score > 0:
                        print(f"[LOG] Rejected weak signal substitution for '{word}' -> '{entry['canonical_term']}' (Score: {score:.5f} < {INFERENCE_THRESHOLD})")
                    else:
                        print(f"[LOG] Negative/zero context for '{word}' -> '{entry['canonical_term']}' (Score: {score:.5f})")
                        
        if best_candidate:
            result = result[:token.start()] + best_candidate['canonical_term'] + result[token.end():]
                    
    return result

In [ ]:
if __name__ == "__main__":
    in_memory_db = {}
    
    # --- TRAINING ---
    in_memory_db = learn_from_observation("ask aditya to review the service", "Ask Aaditya to review the service", in_memory_db)
    in_memory_db = learn_from_observation("aditya is here", "Aaditya is here", in_memory_db)

    print("Memory for 'aditya'->'Aaditya':", json.dumps(in_memory_db.get(f"{get_phonetic_key('aditya')}::Aaditya"), indent=2))
    in_memory_db = learn_from_observation("hi aditya", "hi Aaditya", in_memory_db)
    print("Memory for 'aditya'->'Aaditya':", json.dumps(in_memory_db.get(f"{get_phonetic_key('aditya')}::Aaditya"), indent=2))
    
    # test_1 = "ask aditya to review the service"
    # print(f"Original: {test_1}\nResult:   {apply_memory(test_1, in_memory_db)}\n")
    
    in_memory_db = learn_from_observation("Kiwi is a product and Kiwi is a fruit.", "Kivi is a product and Kiwi is a fruit.", in_memory_db)
    in_memory_db = learn_from_observation("review the sarvam kiwi service", "review the sarvam Kivi service", in_memory_db)
    in_memory_db = learn_from_observation("the kiwi product launch", "the Kivi product launch", in_memory_db)
    
    in_memory_db = learn_from_observation("talk to karthik about it", "talk to Karthick about it", in_memory_db)
    in_memory_db = learn_from_observation("karthik said yes", "Karthick said yes", in_memory_db)
    in_memory_db = learn_from_observation("call karthik", "call Karthick", in_memory_db)

    in_memory_db = learn_from_observation("kiwi is good", "Kivi is good", in_memory_db)

    # Train Kavi (Collision on 'KW' phonetic key with Kivi)
    in_memory_db = learn_from_observation("kiwi is a poet", "Kavi is a poet", in_memory_db)
    in_memory_db = learn_from_observation("the kiwi poem", "the Kavi poem", in_memory_db)
    in_memory_db = learn_from_observation("kiwi writes well", "Kavi writes well", in_memory_db)
    
    # --- INFERENCE EVALUATIONS ---
    print("\n=== Reproducible Evaluation Suite ===\n")
    
    print("-- 1. Unambiguous Proper Nouns --")
    test_1 = "ask aditya to review the service"
    print(f"Original: {test_1}\nResult:   {apply_memory(test_1, in_memory_db)}\n")
    
    test_1b = "call karthik right now"
    print(f"Original: {test_1b}\nResult:   {apply_memory(test_1b, in_memory_db)}\n")
    
    print("-- 2. Positive Context --")
    test_2 = "review the sarvam kiwi service"
    print(f"Original: {test_2}\nResult:   {apply_memory(test_2, in_memory_db)}\n")
    
    test_2b = "the new kiwi product"
    print(f"Original: {test_2b}\nResult:   {apply_memory(test_2b, in_memory_db)}\n")
    
    print("-- 3. Negative Context (Deliberate Inaction) --")
    test_3 = "kiwi for breakfast"
    print(f"Original: {test_3}\nResult:   {apply_memory(test_3, in_memory_db)}\n")
    
    test_3b = "kiwi is a fruit"
    print(f"Original: {test_3b}\nResult:   {apply_memory(test_3b, in_memory_db)}\n")
    
    print("-- 4. Weak Signal Rejection --")
    test_4 = "that kiwi was good"
    print(f"Original: {test_4}\nResult:   {apply_memory(test_4, in_memory_db)}\n")

    print("\n-- 5. Collision Resolution (Kivi vs Kavi) --")
    test_5 = "kiwi writes a poem"
    print(f"Original: {test_5}\nResult:   {apply_memory(test_5, in_memory_db)}\n")
    
    print("\n--- System Memory States ---")
    print("Memory for 'aditya'->'Aaditya':", json.dumps(in_memory_db.get(f"{get_phonetic_key('aditya')}::Aaditya"), indent=2))
    print("Memory for 'kiwi'->'Kivi':", json.dumps(in_memory_db.get(f"{get_phonetic_key('kiwi')}::Kivi"), indent=2))
    print("Memory for 'kiwi'->'Kavi':", json.dumps(in_memory_db.get(f"{get_phonetic_key('kiwi')}::Kavi"), indent=2))


In [6]:
def simulate_bulk_training(history, memory_db):
    print("\n=== Simulating Chronological Bulk Training ===")
    for i, row in enumerate(history):
        raw_llm = row['llm']
        final_user = row['user']
        
        # 1. Simulate the system's inference on the raw LLM output
        memory_applied = apply_memory(raw_llm, memory_db)
        
        # 2. Learn from the diff between what the system outputted and what the user wanted
        memory_db = learn_from_observation(memory_applied, final_user, memory_db)
        
        # 3. Log what happened for visibility
        if memory_applied != final_user:
            if memory_applied == raw_llm:
                print(f"Row {i+1}: User taught the system a NEW correction.")
                print(f"  System output: '{memory_applied}'")
                print(f"  User changed to: '{final_user}'")
            else:
                print(f"Row {i+1}: REVERT DETECTED! System intervened, but user was fed up and reverted it.")
                print(f"  Raw LLM: '{raw_llm}'")
                print(f"  System output: '{memory_applied}'")
                print(f"  User changed to: '{final_user}'")
        else:
            print(f"Row {i+1}: System output matched user expectation perfectly. ('{final_user}')")
            
    return memory_db

# Let's create a fresh memory DB and run the fed-up user scenario!
fresh_db = {}
historical_interactions = [
    {"llm": "kiwi product", "user": "Kivi product"}, # 1. Teach Kivi
    {"llm": "kiwi product", "user": "Kivi product"}, # 2. Teach Kivi
    {"llm": "kiwi product", "user": "Kivi product"}, # 3. Kivi becomes ACTIVE
    {"llm": "kiwi product is beautiful", "user": "kiwi product is beautiful"}, # 4. Fed up user leaves it as kiwi!
]

fresh_db = simulate_bulk_training(historical_interactions, fresh_db)

print("\n--- Memory State After Simulation ---")
print(json.dumps(fresh_db, indent=2))



=== Simulating Chronological Bulk Training ===
Row 1: User taught the system a NEW correction.
  System output: 'kiwi product'
  User changed to: 'Kivi product'
Row 2: User taught the system a NEW correction.
  System output: 'kiwi product'
  User changed to: 'Kivi product'
Row 3: User taught the system a NEW correction.
  System output: 'kiwi product'
  User changed to: 'Kivi product'
Row 4: REVERT DETECTED! System intervened, but user was fed up and reverted it.
  Raw LLM: 'kiwi product is beautiful'
  System output: 'Kivi product is beautiful'
  User changed to: 'kiwi product is beautiful'

--- Memory State After Simulation ---
{
  "KW::Kivi": {
    "canonical_term": "Kivi",
    "phonetic_key": "KW",
    "confidence": 0.3333333333333333,
    "observation_count": 1,
    "ambiguity_risk": true,
    "positive_anchors": {
      "product": 3
    },
    "negative_anchors": {
      "product": 1,
      "beautiful": 1
    },
    "status": "candidate"
  }
}
